# Черновик итоговой аналитической работы

Этот notebook помогает собрать основу итоговой аналитической работы:

1. описание кейса;
2. загрузка данных;
3. консолидация таблиц;
4. проверка качества данных;
5. очистка и preprocessing;
6. описательная статистика;
7. EDA и визуализации;
8. анализ взаимосвязей;
9. подготовка датасета для BI;
10. предварительные выводы.

Работайте последовательно сверху вниз. После каждого крупного блока фиксируйте выводы в markdown-ячейках.

## 1. Описание кейса

Заполните этот блок своими словами.

**Название кейса:**  
[вставьте название]

**Источник данных:**  
[укажите источник: учебные данные, Kaggle, собственные данные, открытый источник]

**Краткое описание данных:**  
[что описывают строки и столбцы]

**Какая аналитическая задача решается:**  
[например: понять, какие категории дают больше выручки и какие факторы связаны с возвратами]

**Вопросы анализа:**
1. [вопрос 1]
2. [вопрос 2]
3. [вопрос 3]

**Основные показатели:**  
[например: выручка, количество, скидка, срок доставки, рейтинг]

**Ограничения данных:**  
[например: нет себестоимости, нет рекламных расходов, часть оценок клиентов отсутствует]

## 2. Проверка окружения и структуры проекта

Сначала проверим, из какой папки открыт notebook и видны ли исходные файлы.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

print("Версия Python:", sys.version)
print("Рабочая папка:", PROJECT_ROOT)
print("Папка исходных данных:", RAW_DIR)
print("Папка результатов:", OUTPUTS_DIR)

required_files = [
    RAW_DIR / "orders_big.csv",
    RAW_DIR / "clients.csv",
    RAW_DIR / "products.csv",
]

print("\nПроверяем файлы:")
for file_path in required_files:
    if file_path.exists():
        print("OK:", file_path)
    else:
        print("НЕ НАЙДЕН:", file_path)

### Что делать, если файл не найден

1. Проверьте, что notebook открыт из корня проекта.
2. Проверьте, что файлы лежат в папке `data/raw`.
3. Проверьте, что имена файлов совпадают с именами в коде.
4. Если вы работаете в Google Colab, загрузите файлы заново после перезапуска среды.

## 3. Импорт библиотек

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", "{:,.2f}".format)

print("pandas:", pd.__version__)
print("numpy:", np.__version__)

## 4. Загрузка основного датасета

В учебном кейсе основной файл называется `orders_big.csv`.  
Если вы используете свой датасет, замените имя файла и список колонок под вашу структуру данных.

In [ ]:
orders_path = RAW_DIR / "orders_big.csv"

if not orders_path.exists():
    raise FileNotFoundError(
        f"Файл не найден: {orders_path}\n"
        "Положите файл orders_big.csv в папку data/raw или измените путь в коде."
    )

orders = pd.read_csv(orders_path)

print("Размер таблицы:", orders.shape)
display(orders.head())

## 5. Первичный обзор данных

Нужно понять:
- сколько строк и столбцов в таблице;
- какие есть типы данных;
- какие признаки числовые;
- какие признаки категориальные;
- есть ли дата события;
- какие потенциальные проблемы уже видны.

In [ ]:
print("Размер таблицы:", orders.shape)

print("\nТипы данных:")
display(orders.dtypes)

print("\nПервые строки:")
display(orders.head())

print("\nПоследние строки:")
display(orders.tail())

memory_mb = orders.memory_usage(deep=True).sum() / 1024 / 1024
print(f"Объём DataFrame в памяти: {memory_mb:.2f} MB")

### Первичный обзор данных

Заполните после выполнения кода.

- Количество строк:
- Количество столбцов:
- Основные сущности:
- Числовые показатели:
- Категориальные признаки:
- Дата или период:
- Возможные проблемы:

## 6. Аккуратная загрузка большого файла

Если файл большой, полезно загружать только нужные колонки и сразу преобразовывать дату.

В учебном кейсе используется набор колонок ниже. Если в вашем датасете другие названия, измените список `usecols`.

In [ ]:
expected_usecols = [
    "order_id",
    "order_date",
    "client_id",
    "product_id",
    "region",
    "channel",
    "category",
    "quantity",
    "unit_price",
    "discount",
    "delivery_days",
    "rating",
    "is_returned",
    "revenue",
]

available_usecols = [col for col in expected_usecols if col in orders.columns]
missing_usecols = [col for col in expected_usecols if col not in orders.columns]

print("Будут использованы колонки:", available_usecols)
print("Не найдены колонки:", missing_usecols)

parse_dates = ["order_date"] if "order_date" in available_usecols else None

orders = pd.read_csv(
    orders_path,
    usecols=available_usecols if available_usecols else None,
    parse_dates=parse_dates
)

print("Размер после выборочной загрузки:", orders.shape)
display(orders.head())
display(orders.dtypes)

memory_mb = orders.memory_usage(deep=True).sum() / 1024 / 1024
print(f"Объём DataFrame в памяти после выборочной загрузки: {memory_mb:.2f} MB")

## 7. Консолидация данных

Если кейс состоит из нескольких таблиц, объедините их в одну аналитическую таблицу.

В учебном кейсе:
- `orders_big.csv` — заказы;
- `clients.csv` — справочник клиентов;
- `products.csv` — справочник товаров.

Если в вашей итоговой работе одна таблица, оставьте `orders_full = orders.copy()`.

In [ ]:
orders_full = orders.copy()

clients_path = RAW_DIR / "clients.csv"
products_path = RAW_DIR / "products.csv"

if clients_path.exists() and "client_id" in orders_full.columns:
    clients = pd.read_csv(clients_path)
    print("clients:", clients.shape)
    display(clients.head())

    before_rows = len(orders_full)
    orders_full = orders_full.merge(clients, on="client_id", how="left")
    after_rows = len(orders_full)

    print("Строк до объединения с clients:", before_rows)
    print("Строк после объединения с clients:", after_rows)
else:
    print("clients.csv не найден или нет client_id. Шаг объединения с клиентами пропущен.")

if products_path.exists() and "product_id" in orders_full.columns:
    products = pd.read_csv(products_path)
    print("products:", products.shape)
    display(products.head())

    before_rows = len(orders_full)
    orders_full = orders_full.merge(products, on="product_id", how="left")
    after_rows = len(orders_full)

    print("Строк до объединения с products:", before_rows)
    print("Строк после объединения с products:", after_rows)
else:
    print("products.csv не найден или нет product_id. Шаг объединения с товарами пропущен.")

print("Итоговый размер аналитической таблицы:", orders_full.shape)
display(orders_full.head())

### Описание консолидации

Заполните после объединения таблиц.

- Какие таблицы были объединены:
- По каким ключам выполнялось объединение:
- Сколько строк было до объединения:
- Сколько строк стало после объединения:
- Были ли потеряны или размножены строки:
- Какие поля добавились после объединения:

## 8. Проверка качества данных

Проверим:
- типы данных;
- количество пропусков;
- долю пропусков;
- количество уникальных значений;
- дубликаты;
- базовые бизнес-правила.

In [ ]:
quality_report = pd.DataFrame({
    "column": orders_full.columns,
    "dtype": orders_full.dtypes.astype(str).values,
    "missing_count": orders_full.isna().sum().values,
    "missing_share_percent": (orders_full.isna().mean().values * 100).round(2),
    "unique_count": orders_full.nunique(dropna=True).values
})

display(quality_report)

quality_report.to_csv(OUTPUTS_DIR / "data_quality_report.csv", index=False)
print("Отчёт качества сохранён:", OUTPUTS_DIR / "data_quality_report.csv")

In [ ]:
checks = {}

if "order_id" in orders_full.columns:
    checks["duplicates_order_id"] = int(orders_full["order_id"].duplicated().sum())

if "quantity" in orders_full.columns:
    checks["negative_quantity"] = int((orders_full["quantity"] < 0).sum())

if "discount" in orders_full.columns:
    checks["discount_less_0"] = int((orders_full["discount"] < 0).sum())
    checks["discount_more_1"] = int((orders_full["discount"] > 1).sum())

if "revenue" in orders_full.columns:
    checks["negative_revenue"] = int((orders_full["revenue"] < 0).sum())

checks

### Вывод по качеству данных

Заполните после проверки.

- Какие поля содержат больше всего пропусков:
- Есть ли дубликаты:
- Есть ли невозможные значения:
- Какие поля нужно преобразовать:
- Какие ограничения нужно указать в итоговой работе:

## 9. Очистка и preprocessing

В этом блоке создаём очищенную версию данных.  
Не изменяйте исходную таблицу без копии.

In [ ]:
orders_clean = orders_full.copy()

# Нормализация текстовых полей
for col in ["channel", "region", "category"]:
    if col in orders_clean.columns:
        orders_clean[col] = orders_clean[col].astype(str).str.strip()
        if col == "channel":
            orders_clean[col] = orders_clean[col].str.lower()

# Преобразование даты
if "order_date" in orders_clean.columns:
    orders_clean["order_date"] = pd.to_datetime(orders_clean["order_date"], errors="coerce")

# Фильтрация невозможных значений
initial_rows = len(orders_clean)

if "quantity" in orders_clean.columns:
    orders_clean = orders_clean[orders_clean["quantity"] > 0]

if "discount" in orders_clean.columns:
    orders_clean = orders_clean[
        (orders_clean["discount"] >= 0) &
        (orders_clean["discount"] <= 1)
    ]

if "revenue" in orders_clean.columns:
    orders_clean = orders_clean[orders_clean["revenue"] >= 0]

print("Строк до очистки:", initial_rows)
print("Строк после очистки:", len(orders_clean))
print("Удалено строк:", initial_rows - len(orders_clean))

display(orders_clean.head())

In [ ]:
# Оптимизация типов для категориальных полей
for col in ["region", "channel", "category"]:
    if col in orders_clean.columns:
        orders_clean[col] = orders_clean[col].astype("category")

if "is_returned" in orders_clean.columns:
    orders_clean["is_returned"] = orders_clean["is_returned"].astype("int8")

memory_mb = orders_clean.memory_usage(deep=True).sum() / 1024 / 1024
print(f"Объём очищенного DataFrame в памяти: {memory_mb:.2f} MB")

processed_path = PROCESSED_DIR / "orders_clean.csv"
orders_clean.to_csv(processed_path, index=False)
print("Очищенный датасет сохранён:", processed_path)

### Описание preprocessing

Заполните после очистки.

- Какие поля были преобразованы:
- Какие строки были удалены:
- Почему эти строки считались некорректными:
- Как изменилась размерность таблицы:
- Какие решения могут повлиять на итоговые выводы:

## 10. Описательная статистика

Рассчитаем основные статистики для числовых показателей.

In [ ]:
numeric_cols = orders_clean.select_dtypes(include=["number"]).columns.tolist()

print("Числовые колонки:")
print(numeric_cols)

descriptive_statistics = orders_clean[numeric_cols].describe().T
display(descriptive_statistics)

descriptive_statistics.to_csv(OUTPUTS_DIR / "descriptive_statistics.csv")
print("Описательная статистика сохранена:", OUTPUTS_DIR / "descriptive_statistics.csv")

### Вывод по описательной статистике

Заполните после расчёта.

- Какие показатели имеют большой разброс:
- Где среднее заметно отличается от медианы:
- Есть ли подозрительные минимумы или максимумы:
- Какие показатели нужно проверить графиками:

## 11. Групповой анализ

Сравним показатели по ключевым категориям.  
В учебном кейсе используются `region`, `channel`, `category`.  
Если в вашем датасете другие признаки, замените список `group_cols`.

In [ ]:
default_group_cols = ["region", "channel", "category"]
group_cols = [col for col in default_group_cols if col in orders_clean.columns]

if not group_cols:
    raise ValueError(
        "Не найдены стандартные группировочные колонки. "
        "Укажите свои категориальные поля в переменной group_cols."
    )

agg_dict = {}

if "order_id" in orders_clean.columns:
    agg_dict["orders_count"] = ("order_id", "nunique")
else:
    agg_dict["rows_count"] = (orders_clean.columns[0], "count")

if "revenue" in orders_clean.columns:
    agg_dict["total_revenue"] = ("revenue", "sum")
    agg_dict["avg_revenue"] = ("revenue", "mean")
    agg_dict["median_revenue"] = ("revenue", "median")

if "delivery_days" in orders_clean.columns:
    agg_dict["avg_delivery_days"] = ("delivery_days", "mean")

if "rating" in orders_clean.columns:
    agg_dict["avg_rating"] = ("rating", "mean")

if "is_returned" in orders_clean.columns:
    agg_dict["return_rate"] = ("is_returned", "mean")

group_summary = (
    orders_clean
    .groupby(group_cols, observed=True)
    .agg(**agg_dict)
    .reset_index()
)

if "total_revenue" in group_summary.columns:
    group_summary = group_summary.sort_values("total_revenue", ascending=False)

display(group_summary.head(20))

group_summary.to_csv(OUTPUTS_DIR / "group_summary.csv", index=False)
print("Групповой отчёт сохранён:", OUTPUTS_DIR / "group_summary.csv")

### Вывод по группам

Ответьте на вопросы:

1. Какие группы дают наибольший вклад в основной показатель?
2. Где выше среднее значение?
3. Где заметны проблемные показатели?
4. Какие группы стоит вынести в BI-дашборд?

## 12. Визуализации для EDA

Каждый график должен отвечать на аналитический вопрос.

In [ ]:
# График топ-групп по выручке
if "region" in orders_clean.columns and "revenue" in orders_clean.columns:
    top_regions = (
        orders_clean
        .groupby("region", observed=True)
        .agg(total_revenue=("revenue", "sum"))
        .reset_index()
        .sort_values("total_revenue", ascending=False)
        .head(10)
    )

    plt.figure(figsize=(10, 5))
    plt.bar(top_regions["region"].astype(str), top_regions["total_revenue"])
    plt.title("Топ-10 регионов по выручке")
    plt.xlabel("Регион")
    plt.ylabel("Выручка")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
else:
    print("Для графика нужны поля region и revenue.")

In [ ]:
# Распределение выручки
if "revenue" in orders_clean.columns:
    plt.figure(figsize=(8, 5))
    plt.hist(orders_clean["revenue"].dropna(), bins=50)
    plt.title("Распределение выручки по заказам")
    plt.xlabel("Выручка")
    plt.ylabel("Количество наблюдений")
    plt.tight_layout()
    plt.show()
else:
    print("Поле revenue не найдено. Выберите другой числовой показатель.")

In [ ]:
# Boxplot для срока доставки
if "delivery_days" in orders_clean.columns:
    plt.figure(figsize=(8, 5))
    plt.boxplot(orders_clean["delivery_days"].dropna())
    plt.title("Распределение сроков доставки")
    plt.ylabel("Дни доставки")
    plt.tight_layout()
    plt.show()
else:
    print("Поле delivery_days не найдено. Выберите другой числовой показатель.")

### Выводы по визуализациям

Заполните после построения графиков.

**График 1:**  
- Что показывает:
- Какой вывод:
- Ограничение:

**График 2:**  
- Что показывает:
- Какой вывод:
- Ограничение:

**График 3:**  
- Что показывает:
- Какой вывод:
- Ограничение:

## 13. Анализ взаимосвязей

Корреляция помогает увидеть статистическую связь между числовыми показателями.  
Важно: корреляция не доказывает причинно-следственную связь.

In [ ]:
corr_cols = []

for col in ["quantity", "unit_price", "discount", "delivery_days", "rating", "is_returned", "revenue"]:
    if col in orders_clean.columns and pd.api.types.is_numeric_dtype(orders_clean[col]):
        corr_cols.append(col)

if len(corr_cols) < 2:
    raise ValueError("Для корреляции нужно минимум два числовых столбца.")

print("Колонки для корреляции:", corr_cols)

correlation_matrix = orders_clean[corr_cols].corr(method="pearson")
display(correlation_matrix)

correlation_matrix.to_csv(OUTPUTS_DIR / "correlation_matrix.csv")
print("Корреляционная матрица сохранена:", OUTPUTS_DIR / "correlation_matrix.csv")

In [ ]:
# Scatter plot: скидка и выручка
if "discount" in orders_clean.columns and "revenue" in orders_clean.columns:
    sample_orders = orders_clean.sample(min(3000, len(orders_clean)), random_state=42)

    plt.figure(figsize=(8, 5))
    plt.scatter(sample_orders["discount"], sample_orders["revenue"], alpha=0.3)
    plt.title("Связь скидки и выручки")
    plt.xlabel("Скидка")
    plt.ylabel("Выручка")
    plt.tight_layout()
    plt.show()
else:
    print("Для scatter plot нужны поля discount и revenue.")

In [ ]:
# Scatter plot: срок доставки и рейтинг
if "delivery_days" in orders_clean.columns and "rating" in orders_clean.columns:
    sample_orders = orders_clean.sample(min(3000, len(orders_clean)), random_state=42)

    plt.figure(figsize=(8, 5))
    plt.scatter(sample_orders["delivery_days"], sample_orders["rating"], alpha=0.3)
    plt.title("Связь срока доставки и рейтинга")
    plt.xlabel("Срок доставки, дней")
    plt.ylabel("Рейтинг")
    plt.tight_layout()
    plt.show()
else:
    print("Для scatter plot нужны поля delivery_days и rating.")

### Вывод по взаимосвязям

Ответьте на вопросы:

1. Какие показатели связаны между собой сильнее всего?
2. Есть ли связь между скидкой и выручкой?
3. Есть ли связь между сроком доставки и рейтингом?
4. Можно ли считать найденные связи причинно-следственными?
5. Какие дополнительные проверки нужны?

## 14. Подготовка датасета для BI

Сохраним чистый датасет, который можно загрузить в Power BI, Tableau Public, Yandex DataLens или другую BI-платформу.

In [ ]:
bi_dataset = orders_clean.copy()

if "order_date" in bi_dataset.columns:
    bi_dataset["order_date"] = pd.to_datetime(bi_dataset["order_date"], errors="coerce")
    bi_dataset["order_month"] = bi_dataset["order_date"].dt.to_period("M").astype(str)

if "revenue" in bi_dataset.columns and "quantity" in bi_dataset.columns:
    bi_dataset["revenue_per_item"] = (
        bi_dataset["revenue"] / bi_dataset["quantity"].replace(0, pd.NA)
    )

bi_path = OUTPUTS_DIR / "bi_dataset.csv"
bi_dataset.to_csv(bi_path, index=False)

print("BI-датасет сохранён:", bi_path)
print("Размер BI-датасета:", bi_dataset.shape)
display(bi_dataset.head())

### Что проверить перед загрузкой в BI

- Есть ли понятные названия столбцов.
- Есть ли дата или период для динамики.
- Есть ли категориальные поля для фильтров.
- Есть ли числовые показатели для KPI.
- Нет ли явных дублей и невозможных значений.
- Сохранён ли файл `outputs/bi_dataset.csv`.

## 15. Предварительные выводы

Заполните выводы своими словами. Каждый вывод должен опираться на расчёты, таблицы или графики.

In [ ]:
conclusions = '''
# Предварительные выводы

## 1. Что анализировалось
[заполните своими словами]

## 2. Какие данные использовались
[заполните своими словами]

## 3. Какие проверки качества выполнены
[заполните своими словами]

## 4. Основные закономерности
[заполните своими словами]

## 5. Найденные взаимосвязи
[заполните своими словами]

## 6. Ограничения анализа
[заполните своими словами]

## 7. Что рекомендуется проверить дальше
[заполните своими словами]
'''

conclusions_path = OUTPUTS_DIR / "preliminary_conclusions.md"

with open(conclusions_path, "w", encoding="utf-8") as file:
    file.write(conclusions)

print("Файл с предварительными выводами создан:", conclusions_path)

## 16. Как заменить учебные данные на свой датасет

| В учебном кейсе | В вашей итоговой работе |
|---|---|
| `orders_big.csv` | ваш основной CSV-файл |
| `clients.csv` | ваш справочник клиентов или объектов |
| `products.csv` | ваш справочник товаров, услуг или категорий |
| `order_date` | дата события |
| `client_id` | идентификатор клиента, пользователя или объекта |
| `region`, `channel`, `category` | ваши категориальные признаки |
| `revenue`, `quantity`, `rating` | ваши числовые показатели |
| `bi_dataset.csv` | подготовленный файл для BI |
| `preliminary_conclusions.md` | черновик заключения |

Не копируйте выводы учебного кейса. Переносите структуру работы.

## 17. Финальный чек-лист

Проверьте перед завершением работы:

- [ ] Проект открыт из правильной папки.
- [ ] Исходные файлы найдены.
- [ ] Данные загружены.
- [ ] Таблицы объединены, если было несколько источников.
- [ ] Проверены типы данных.
- [ ] Проверены пропуски.
- [ ] Проверены дубликаты.
- [ ] Проверены аномальные значения.
- [ ] Сохранён `data_quality_report.csv`.
- [ ] Сохранён `descriptive_statistics.csv`.
- [ ] Сохранён `group_summary.csv`.
- [ ] Построены EDA-графики.
- [ ] Сохранён `correlation_matrix.csv`.
- [ ] Подготовлен `bi_dataset.csv`.
- [ ] Создан `preliminary_conclusions.md`.
- [ ] Понятно, как заменить учебные данные на свои.

## 18. Следующие шаги для полной итоговой работы

После этого notebook нужно дополнить:

1. BI-дашбордом на основе `bi_dataset.csv`;
2. ABC-анализом;
3. XYZ-анализом;
4. RFM-анализом, если в данных есть клиенты, даты и суммы покупок;
5. итоговым заключением по всей работе;
6. дополнительным блоком ML по желанию.